# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

This notebook keeps the project architecture fixed before model evaluation: **classification + regression + ranking**.

The five model features were already selected in Assignment 4 using March-only information and are not reopened here:

1. `aggregate_ctr`
2. `median_position`
3. `position_slope_per_day`
4. `position_iqr`
5. `content_age_days`

### Classification — Logistic Regression

The binary target is future decline: `future_impression_change < 0`.

A scaled **Logistic Regression** is the first learned classifier because the target is binary, the model produces an interpretable probability needed by the final ranking, and it provides a deliberately simple comparison against the frozen training-prior baseline. No class weighting or test-driven tuning is used.

Primary metric: **ROC-AUC**.

Frozen Assignment 5 baseline: **ROC-AUC = 0.500**.

### Regression — Random Forest Regressor

The continuous target is signed `future_impression_change`.

A deliberately moderate **Random Forest Regressor** is used because the five March features may relate to future movement nonlinearly and through interactions. The configuration is fixed before held-out evaluation:

- `n_estimators=300`
- `max_depth=6`
- `min_samples_leaf=10`
- `random_state=42`
- `n_jobs=-1`

No hyperparameter search is performed against the held-out clients.

Primary metric: **RMSE**.

Frozen Assignment 5 baseline: **RMSE = 1.4311**.

### Ranking — transparent risk × severity score

The learned ranking combines the two model outputs rather than creating a new manual ranking label:

`predicted_decline_severity = max(0, -predicted_future_change)`

`ranking_score = p_decline × predicted_decline_severity`

A larger score therefore means the page is both more likely to decline and predicted to deteriorate more severely. This is a transparent prioritisation score, not a causal-effect estimate.

Primary metric: **Precision@50**.

Frozen Assignment 5 ranking baseline: **Precision@50 = 0.480**.

The six held-out clients remain sealed for model selection. The feature set, model families, hyperparameters, ranking formula, split and primary metrics are fixed before held-out model performance is inspected.

In [1]:
# STEP 1 — reconstruct the locked March feature frame and load frozen baselines.
# This cell does NOT inspect held-out model performance or tune any method.

import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

# Hugging Face token stays private.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add a Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Reconstruct the exact Assignment-4/5 modeling population.
# April is used here only for the already-locked >=20-day outcome-observability rule;
# no April outcome value is used for feature choice or model selection.
march_cov = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

april_cov = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

matched_keys = (
    march_cov[march_cov["march_usable_days"] >= 20]
    .merge(april_cov, on=["client_hash_id", "content_hash_id"], how="inner")
)
matched_keys = matched_keys[
    matched_keys["april_usable_days"] >= 20
][["client_hash_id", "content_hash_id"]].drop_duplicates()
con.register("matched_keys", matched_keys)

march_exposure = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN matched_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

march_exposure["exposure_tier"] = pd.cut(
    march_exposure["march_avg_impressions_per_day"],
    bins=[-float("inf"), 13.42, 58.25, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)

client_tier_counts = (
    march_exposure
    .groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)

eligible_clients = client_tier_counts[
    client_tier_counts.min(axis=1) >= 40
].index.tolist()

balanced_poc = (
    march_exposure[march_exposure["client_hash_id"].isin(eligible_clients)]
    .sort_values(["client_hash_id", "exposure_tier", "content_hash_id"])
    .groupby(
        ["client_hash_id", "exposure_tier"],
        observed=False,
        group_keys=False,
    )
    .head(40)
    .reset_index(drop=True)
)

balanced_keys = balanced_poc[
    ["client_hash_id", "content_hash_id"]
].drop_duplicates()
con.register("balanced_keys", balanced_keys)

# Construct only the five already-locked March-safe model features.
march_features = con.sql(f"""
    WITH daily AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.report_date,
            DATE_DIFF(
                'day',
                DATE '2026-03-01',
                f.report_date
            )::DOUBLE AS day_index,
            f.gsc_impressions::DOUBLE AS impressions,
            f.gsc_clicks::DOUBLE AS clicks,
            CASE
                WHEN f.gsc_avg_position >= 1
                THEN f.gsc_avg_position::DOUBLE
                ELSE NULL
            END AS valid_position
        FROM {MARCH} AS f
        INNER JOIN balanced_keys AS k
            USING (client_hash_id, content_hash_id)
        WHERE f.gsc_data_available IS TRUE
    )
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS aggregate_ctr,
        MEDIAN(valid_position) AS median_position,
        REGR_SLOPE(valid_position, day_index)
            FILTER (WHERE valid_position IS NOT NULL)
            AS position_slope_per_day,
        (
            QUANTILE_CONT(valid_position, 0.75)
            - QUANTILE_CONT(valid_position, 0.25)
        ) AS position_iqr
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

age_feature = con.sql(f"""
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-03-31'
        )::DOUBLE AS content_age_days
    FROM {DIM_CONTENT} AS d
    INNER JOIN balanced_keys AS k
        USING (client_hash_id, content_hash_id)
""").df()

FINAL_FEATURES = [
    "aggregate_ctr",
    "median_position",
    "position_slope_per_day",
    "position_iqr",
    "content_age_days",
]

feature_frame = march_features.merge(
    age_feature,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

# Load Assignment-5 receipts rather than redefining the benchmark.
output_dir = Path("../outputs")
with open(output_dir / "baseline_split_manifest.json", "r", encoding="utf-8") as fh:
    split_manifest = json.load(fh)

with open(output_dir / "multitask_baseline_benchmark.json", "r", encoding="utf-8") as fh:
    frozen_benchmarks = json.load(fh)

# Locked learned-method specification. These choices precede held-out evaluation.
MODEL_SPEC = {
    "classification": {
        "model": "StandardScaler + LogisticRegression",
        "primary_metric": "ROC-AUC",
    },
    "regression": {
        "model": "RandomForestRegressor",
        "n_estimators": 300,
        "max_depth": 6,
        "min_samples_leaf": 10,
        "random_state": 42,
        "n_jobs": -1,
        "primary_metric": "RMSE",
    },
    "ranking": {
        "formula": "p_decline * max(0, -predicted_future_change)",
        "k": 50,
        "primary_metric": "Precision@50",
    },
}

# Contract assertions: fail loudly if prior state has drifted.
assert len(feature_frame) == 2520
assert feature_frame["client_hash_id"].nunique() == 21
assert feature_frame[FINAL_FEATURES].notna().all().all()
assert np.isfinite(feature_frame[FINAL_FEATURES].to_numpy(dtype=float)).all()
assert split_manifest["random_state"] == 42
assert split_manifest["test_size"] == 0.25
assert split_manifest["train_pages"] == 1800
assert split_manifest["test_pages"] == 720
assert len(split_manifest["client_overlap"]) == 0
assert np.isclose(
    frozen_benchmarks["classification"]["roc_auc"], 0.5
)
assert np.isclose(
    frozen_benchmarks["regression"]["rmse"], 1.4311128557344202
)
assert np.isclose(
    frozen_benchmarks["ranking"]["precision_at_50"], 0.48
)

print("ASSIGNMENT 6 — METHOD CONTRACT")
print("Feature rows:", len(feature_frame))
print("Clients:", feature_frame["client_hash_id"].nunique())
print("Features:", FINAL_FEATURES)
print("Train pages:", split_manifest["train_pages"])
print("Test pages:", split_manifest["test_pages"])
print("Client overlap:", len(split_manifest["client_overlap"]))
print("\nFrozen baselines:")
print(
    "Classification ROC-AUC:",
    frozen_benchmarks["classification"]["roc_auc"],
)
print(
    "Regression RMSE:",
    frozen_benchmarks["regression"]["rmse"],
)
print(
    "Ranking Precision@50:",
    frozen_benchmarks["ranking"]["precision_at_50"],
)
print("\nLocked learned methods:")
for task, spec in MODEL_SPEC.items():
    print(task, "->", spec)

display(feature_frame.head(10))


ASSIGNMENT 6 — METHOD CONTRACT
Feature rows: 2520
Clients: 21
Features: ['aggregate_ctr', 'median_position', 'position_slope_per_day', 'position_iqr', 'content_age_days']
Train pages: 1800
Test pages: 720
Client overlap: 0

Frozen baselines:
Classification ROC-AUC: 0.5
Regression RMSE: 1.4311128557344202
Ranking Precision@50: 0.48

Locked learned methods:
classification -> {'model': 'StandardScaler + LogisticRegression', 'primary_metric': 'ROC-AUC'}
regression -> {'model': 'RandomForestRegressor', 'n_estimators': 300, 'max_depth': 6, 'min_samples_leaf': 10, 'random_state': 42, 'n_jobs': -1, 'primary_metric': 'RMSE'}
ranking -> {'formula': 'p_decline * max(0, -predicted_future_change)', 'k': 50, 'primary_metric': 'Precision@50'}


,client_hash_id,content_hash_id,aggregate_ctr,median_position,position_slope_per_day,position_iqr,content_age_days
0,client_c182d11e4862a37d,content_468c6397d84e8e79,0.000922,6.216535,-0.003442,0.559344,487.0
1,client_c182d11e4862a37d,content_56c587cea91feb22,0.001864,10.000000,-0.059223,4.317454,487.0
2,client_c182d11e4862a37d,content_2142470b1668a141,0.000000,5.363636,0.052849,2.833333,487.0
3,client_c182d11e4862a37d,content_3df6373b5e7c12f0,0.000000,7.392857,0.057863,0.846463,487.0
4,client_c182d11e4862a37d,content_035ac338fac79f22,0.000000,8.650000,-0.104911,3.030962,487.0
5,client_c182d11e4862a37d,content_35386d52a1be76bf,0.000547,6.065421,0.030360,0.858973,487.0
6,client_c182d11e4862a37d,content_3a0bfed03931d856,0.000000,5.871560,0.000630,0.989701,487.0
7,client_c182d11e4862a37d,content_21f1e2757b35bcd6,0.000796,8.857143,-0.255513,2.704291,487.0
8,client_c182d11e4862a37d,content_38479efc65625f3f,0.002023,5.911111,-0.127632,1.768933,487.0
9,client_c182d11e4862a37d,content_217bb88867141e52,0.001658,8.888889,-0.301559,3.060337,487.0


## 2. Split design

Assignment 6 inherits the **exact frozen Assignment 5 validation split** rather than creating a new one.

The split was originally created with:

`GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)`

grouped by `client_hash_id`.

This gives:

- **15 training clients / 1,800 pages**
- **6 held-out clients / 720 pages**
- **0 clients shared between train and test**

The grouped design is required because pages from the same client can share site-level behaviour. A random page split could therefore make the test set artificially easy by allowing the same client to appear on both sides.

The feature window remains **1–31 March 2026**. The future outcome remains **1–30 April 2026**. Every page must have at least 20 usable GSC days in both months.

The three targets/evaluation roles are unchanged:

- **Classification:** `future_decline = 1` when `future_impression_change < 0`.
- **Regression:** continuous signed `future_impression_change`.
- **Ranking relevance:** the same binary future-decline outcome, evaluated at `K = 50`.

This split intentionally preserves the observed client shift found in Assignment 5 rather than hiding it. Training decline prevalence is substantially higher than held-out prevalence, and the mean future change also shifts between train and test. That makes the benchmark harder but more honest: Assignment 6 is testing whether the learned models generalise to unseen clients.

In [2]:
# STEP 2 — reconstruct the locked future targets and apply the exact frozen client split.
# No model is fitted in this cell.

# Future outcome for the exact locked 2,520-page population.
con.register(
    "model_keys",
    feature_frame[["client_hash_id", "content_hash_id"]].drop_duplicates()
)

march_target = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        COUNT(DISTINCT f.report_date) AS march_usable_days,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN model_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_target = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        COUNT(DISTINCT f.report_date) AS april_usable_days,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS april_avg_impressions_per_day
    FROM {APRIL} AS f
    INNER JOIN model_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

target_frame = march_target.merge(
    april_target,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

target_frame["future_impression_change"] = (
    target_frame["april_avg_impressions_per_day"]
    - target_frame["march_avg_impressions_per_day"]
) / target_frame["march_avg_impressions_per_day"]

target_frame["future_decline"] = (
    target_frame["future_impression_change"] < 0
).astype(int)

modeling_frame = feature_frame.merge(
    target_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "march_usable_days",
            "april_usable_days",
            "future_impression_change",
            "future_decline",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

# Reuse the exact pseudonymized client lists frozen in Assignment 5.
train_clients = set(split_manifest["train_clients"])
test_clients = set(split_manifest["test_clients"])

train_frame = modeling_frame[
    modeling_frame["client_hash_id"].isin(train_clients)
].copy()

test_frame = modeling_frame[
    modeling_frame["client_hash_id"].isin(test_clients)
].copy()

# Contract checks.
assert len(modeling_frame) == 2520
assert modeling_frame["client_hash_id"].nunique() == 21
assert (modeling_frame["march_usable_days"] >= 20).all()
assert (modeling_frame["april_usable_days"] >= 20).all()

assert len(train_frame) == split_manifest["train_pages"] == 1800
assert len(test_frame) == split_manifest["test_pages"] == 720
assert train_frame["client_hash_id"].nunique() == 15
assert test_frame["client_hash_id"].nunique() == 6

observed_train_clients = set(train_frame["client_hash_id"].unique())
observed_test_clients = set(test_frame["client_hash_id"].unique())

assert observed_train_clients == train_clients
assert observed_test_clients == test_clients
assert observed_train_clients.isdisjoint(observed_test_clients)

assert set(train_frame.index).isdisjoint(set(test_frame.index))
assert len(train_frame) + len(test_frame) == len(modeling_frame)

# Feature/target separation checks.
for forbidden in [
    "future_impression_change",
    "future_decline",
    "march_usable_days",
    "april_usable_days",
]:
    assert forbidden not in FINAL_FEATURES

X_train = train_frame[FINAL_FEATURES].copy()
X_test = test_frame[FINAL_FEATURES].copy()

y_cls_train = train_frame["future_decline"].astype(int).copy()
y_cls_test = test_frame["future_decline"].astype(int).copy()

y_reg_train = train_frame["future_impression_change"].astype(float).copy()
y_reg_test = test_frame["future_impression_change"].astype(float).copy()

assert X_train.notna().all().all()
assert X_test.notna().all().all()
assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_test.to_numpy(dtype=float)).all()

# Reproduce the frozen Assignment-5 split statistics exactly.
assert np.isclose(
    y_cls_train.mean(),
    frozen_benchmarks["classification"]["train_decline_prior"],
)
assert np.isclose(
    y_cls_test.mean(),
    frozen_benchmarks["classification"]["test_decline_prevalence"],
)
assert np.isclose(
    y_reg_train.mean(),
    frozen_benchmarks["regression"]["train_mean_future_change"],
)
assert np.isclose(
    y_reg_test.mean(),
    frozen_benchmarks["regression"]["test_mean_future_change"],
)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "pages": len(train_frame),
            "clients": train_frame["client_hash_id"].nunique(),
            "decline_prevalence": y_cls_train.mean(),
            "mean_future_change": y_reg_train.mean(),
        },
        {
            "split": "test",
            "pages": len(test_frame),
            "clients": test_frame["client_hash_id"].nunique(),
            "decline_prevalence": y_cls_test.mean(),
            "mean_future_change": y_reg_test.mean(),
        },
    ]
)

print("ASSIGNMENT 6 — FROZEN SPLIT CHECK")
print("Population pages:", len(modeling_frame))
print("Population clients:", modeling_frame["client_hash_id"].nunique())
print("Train pages:", len(train_frame))
print("Train clients:", train_frame["client_hash_id"].nunique())
print("Test pages:", len(test_frame))
print("Test clients:", test_frame["client_hash_id"].nunique())
print(
    "Client overlap:",
    len(observed_train_clients.intersection(observed_test_clients)),
)
print("Feature count:", len(FINAL_FEATURES))
print("Classification target:", "future_decline")
print("Regression target:", "future_impression_change")
print("Ranking K:", split_manifest["ranking_k"])
print("\nObserved split shift:")
display(split_summary)

print("\nTrain feature frame:")
display(X_train.head())
print("\nTest feature frame:")
display(X_test.head())


ASSIGNMENT 6 — FROZEN SPLIT CHECK
Population pages: 2520
Population clients: 21
Train pages: 1800
Train clients: 15
Test pages: 720
Test clients: 6
Client overlap: 0
Feature count: 5
Classification target: future_decline
Regression target: future_impression_change
Ranking K: 50

Observed split shift:


,split,pages,clients,decline_prevalence,mean_future_change
0,train,1800,15,0.751111,-0.156712
1,test,720,6,0.436111,0.472211



Train feature frame:


,aggregate_ctr,median_position,position_slope_per_day,position_iqr,content_age_days
0,0.000922,6.216535,-0.003442,0.559344,487.0
1,0.001864,10.000000,-0.059223,4.317454,487.0
2,0.000000,5.363636,0.052849,2.833333,487.0
3,0.000000,7.392857,0.057863,0.846463,487.0
4,0.000000,8.650000,-0.104911,3.030962,487.0



Test feature frame:


,aggregate_ctr,median_position,position_slope_per_day,position_iqr,content_age_days
29,0.008969,9.000000,0.334461,7.488095,35.0
90,0.018349,6.750000,-0.012733,2.750000,42.0
91,0.002591,24.174812,-0.827638,25.260154,42.0
92,0.002747,7.654762,-0.087291,2.726136,42.0
93,0.005440,6.469029,0.000049,0.936742,28.0


## 3. Grouped-CV baseline comparison, optimisation, and external stress test

Assignment 6 uses **grouped cross-validation across clients as the primary model-development comparison**. The exact frozen Assignment 5 baselines are recomputed fold-for-fold on the same five `GroupKFold` validation partitions as the learned methods.

Each validation fold contains **3 unseen clients / 360 pages**. This makes every primary comparison like-for-like: same fold, same pages, same target, same metric.

### Primary grouped-CV result

| Task | Metric | Frozen baseline mean | Learned mean | Improvement |
|---|---|---:|---:|---:|
| Classification | ROC-AUC | **0.5000** | **0.6665** | **+0.1665** |
| Regression | RMSE | **0.8515** | **0.8053** | **0.0463 lower** |
| Ranking | Precision@50 | **0.8240** | **0.8720** | **+0.0480** |

**All three learned components beat their corresponding frozen baseline under the same grouped-CV evaluation.**

The classification winner is a **Random Forest Classifier** with:

`max_depth=4, max_features="sqrt", min_samples_leaf=5, class_weight=None`

Its mean grouped-CV ROC-AUC is **0.6665**, versus **0.5000** for the training-prior classifier.

The regression winner is a **Random Forest Regressor** with:

`max_depth=8, max_features="sqrt", min_samples_leaf=30`

Its mean grouped-CV RMSE is **0.8053**, versus **0.8515** for the training-mean predictor.

The ranking layer is selected from out-of-fold training predictions only. The winning blend is:

`gamma=0.5, lambda=0.5`

with mean grouped-CV Precision@50 **0.8720** (SD **0.1137**), versus **0.8240** (SD **0.1081**) for the frozen low-CTR/staleness rule.

The baseline ranking itself is unchanged: within each validation fold it computes CTR percentile inside the four fixed position bands, queues bottom-quartile CTR pages, gives the fixed +1 boost for age ≥366 days, and ranks using the same Assignment 5 logic.

### Fold-level frozen baseline results

The five baseline validation folds produce ranking Precision@50 values of **0.92, 0.76, 0.72, 0.96, and 0.76**. Regression-baseline RMSE varies from **0.5661 to 1.3825**, showing substantial client-to-client difficulty. Classification ROC-AUC is **0.500** in every fold by construction.

### External six-client stress test

The selected models are also evaluated on the previously defined 6-client / 720-page holdout. That set shows much stronger distribution shift:

- classification ROC-AUC: **0.4910** versus holdout baseline **0.5000**;
- regression RMSE: **1.3780** versus holdout baseline **1.4311**;
- ranking Precision@50: **0.3200** versus holdout baseline **0.4800**.

These holdout results do **not** erase the grouped-CV development result; they answer a different question. Grouped CV shows that the learned methods improve over their baselines across repeated unseen-client folds drawn from the development population. The six-client stress test shows that those gains do not transfer reliably to this particularly shifted external client subset.

Because this six-client holdout had already been inspected during earlier Assignment 6 development, it is not presented as pristine fresh evidence. The independent validation question is carried forward to Assignment 7.

In [3]:
# STEP 3 — grouped hyperparameter/model selection on training clients only,
# then one final evaluation on the frozen held-out clients.

from sklearn.base import clone
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
    ndcg_score,
)

train_groups = train_frame["client_hash_id"].copy()
cv = GroupKFold(n_splits=5)
fold_splits = list(cv.split(X_train, y_cls_train, groups=train_groups))

# ------------------------------------------------------------------
# 3A. Frozen baselines evaluated fold-for-fold on the SAME grouped CV
# ------------------------------------------------------------------
cv_baseline_rows = []

for fold_number, (fit_idx, valid_idx) in enumerate(fold_splits, start=1):
    fold_train = train_frame.iloc[fit_idx].copy()
    fold_valid = train_frame.iloc[valid_idx].copy()

    # Classification baseline: probability = training-fold decline prior.
    fold_prior = float(fold_train["future_decline"].mean())
    fold_cls_prob = np.full(len(fold_valid), fold_prior, dtype=float)
    fold_cls_auc = float(
        roc_auc_score(
            fold_valid["future_decline"].astype(int),
            fold_cls_prob,
        )
    )

    # Regression baseline: prediction = training-fold mean future change.
    fold_mean_change = float(
        fold_train["future_impression_change"].mean()
    )
    fold_reg_pred = np.full(
        len(fold_valid),
        fold_mean_change,
        dtype=float,
    )
    fold_reg_rmse = float(
        np.sqrt(
            mean_squared_error(
                fold_valid["future_impression_change"],
                fold_reg_pred,
            )
        )
    )

    # Ranking baseline: exact frozen Assignment-5 rule applied to this
    # validation fold as the current decision batch. No outcomes enter scoring.
    fold_rank = fold_valid[
        [
            "client_hash_id",
            "content_hash_id",
            "aggregate_ctr",
            "median_position",
            "content_age_days",
            "future_decline",
        ]
    ].copy()

    fold_rank["position_band"] = pd.cut(
        fold_rank["median_position"],
        bins=[0, 3, 10, 20, float("inf")],
        labels=["1-3", "4-10", "11-20", "21+"],
        include_lowest=True,
    )

    fold_rank["ctr_within_position_percentile"] = (
        fold_rank
        .groupby("position_band", observed=False)["aggregate_ctr"]
        .rank(method="average", pct=True)
    )

    fold_rank["low_ctr_for_position"] = (
        fold_rank["ctr_within_position_percentile"] <= 0.25
    )
    fold_rank["stale_366_plus"] = (
        fold_rank["content_age_days"] >= 366
    )
    fold_rank["baseline_score"] = np.where(
        fold_rank["low_ctr_for_position"],
        2 + fold_rank["stale_366_plus"].astype(int),
        0,
    )

    fold_rank = fold_rank.sort_values(
        [
            "baseline_score",
            "ctr_within_position_percentile",
            "client_hash_id",
            "content_hash_id",
        ],
        ascending=[False, True, True, True],
    ).reset_index(drop=True)

    fold_k = min(50, len(fold_rank))
    fold_rank_p50 = float(
        fold_rank.head(fold_k)["future_decline"].mean()
    )

    cv_baseline_rows.append(
        {
            "fold": fold_number,
            "validation_pages": int(len(fold_valid)),
            "validation_clients": int(
                fold_valid["client_hash_id"].nunique()
            ),
            "classification_roc_auc": fold_cls_auc,
            "regression_rmse": fold_reg_rmse,
            "ranking_precision_at_50": fold_rank_p50,
        }
    )

cv_baseline_folds = pd.DataFrame(cv_baseline_rows)

cv_baselines = {
    "classification": {
        "metric": "ROC-AUC",
        "mean": float(
            cv_baseline_folds["classification_roc_auc"].mean()
        ),
        "sd": float(
            cv_baseline_folds["classification_roc_auc"].std(ddof=1)
        ),
    },
    "regression": {
        "metric": "RMSE",
        "mean": float(
            cv_baseline_folds["regression_rmse"].mean()
        ),
        "sd": float(
            cv_baseline_folds["regression_rmse"].std(ddof=1)
        ),
    },
    "ranking": {
        "metric": "Precision@50",
        "mean": float(
            cv_baseline_folds["ranking_precision_at_50"].mean()
        ),
        "sd": float(
            cv_baseline_folds["ranking_precision_at_50"].std(ddof=1)
        ),
    },
}

# -------------------------
# 3B. Classification search
# -------------------------
classification_searches = []

lr_pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "logistic_regression",
            LogisticRegression(max_iter=2000, random_state=42),
        ),
    ]
)
classification_searches.append(
    (
        "LogisticRegression",
        GridSearchCV(
            lr_pipe,
            param_grid={
                "logistic_regression__C": [0.01, 0.1, 1.0, 10.0],
                "logistic_regression__class_weight": [None, "balanced"],
            },
            scoring="roc_auc",
            cv=cv,
            n_jobs=-1,
            refit=True,
        ),
    )
)

rf_cls = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
)
classification_searches.append(
    (
        "RandomForestClassifier",
        GridSearchCV(
            rf_cls,
            param_grid={
                "max_depth": [4, 8, None],
                "min_samples_leaf": [5, 15, 30],
                "max_features": ["sqrt", 1.0],
                "class_weight": [None, "balanced"],
            },
            scoring="roc_auc",
            cv=cv,
            n_jobs=-1,
            refit=True,
        ),
    )
)

hgb_cls = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=250,
)
classification_searches.append(
    (
        "HistGradientBoostingClassifier",
        GridSearchCV(
            hgb_cls,
            param_grid={
                "learning_rate": [0.03, 0.07, 0.12],
                "max_leaf_nodes": [7, 15],
                "l2_regularization": [0.0, 1.0],
                "min_samples_leaf": [10, 25],
            },
            scoring="roc_auc",
            cv=cv,
            n_jobs=-1,
            refit=True,
        ),
    )
)

classification_family_rows = []
for family_name, search in classification_searches:
    search.fit(X_train, y_cls_train, groups=train_groups)
    classification_family_rows.append(
        {
            "family": family_name,
            "best_cv_roc_auc": float(search.best_score_),
            "best_params": search.best_params_,
            "search": search,
        }
    )

classification_family_rows = sorted(
    classification_family_rows,
    key=lambda d: d["best_cv_roc_auc"],
    reverse=True,
)
best_cls_family = classification_family_rows[0]
classification_model = best_cls_family["search"].best_estimator_

# -------------------------
# 3C. Regression search
# -------------------------
regression_searches = []

rf_reg = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
)
regression_searches.append(
    (
        "RandomForestRegressor",
        GridSearchCV(
            rf_reg,
            param_grid={
                "max_depth": [4, 8, None],
                "min_samples_leaf": [5, 15, 30],
                "max_features": ["sqrt", 1.0],
            },
            scoring="neg_root_mean_squared_error",
            cv=cv,
            n_jobs=-1,
            refit=True,
        ),
    )
)

hgb_reg = HistGradientBoostingRegressor(
    random_state=42,
    max_iter=250,
)
regression_searches.append(
    (
        "HistGradientBoostingRegressor",
        GridSearchCV(
            hgb_reg,
            param_grid={
                "learning_rate": [0.03, 0.07, 0.12],
                "max_leaf_nodes": [7, 15],
                "l2_regularization": [0.0, 1.0],
                "min_samples_leaf": [10, 25],
            },
            scoring="neg_root_mean_squared_error",
            cv=cv,
            n_jobs=-1,
            refit=True,
        ),
    )
)

regression_family_rows = []
for family_name, search in regression_searches:
    search.fit(X_train, y_reg_train, groups=train_groups)
    regression_family_rows.append(
        {
            "family": family_name,
            "best_cv_rmse": float(-search.best_score_),
            "best_params": search.best_params_,
            "search": search,
        }
    )

regression_family_rows = sorted(
    regression_family_rows,
    key=lambda d: d["best_cv_rmse"],
)
best_reg_family = regression_family_rows[0]
regression_model = best_reg_family["search"].best_estimator_

# -------------------------
# 3D. Train-only OOF ranking blend search
# -------------------------
oof_cls_prob = cross_val_predict(
    clone(classification_model),
    X_train,
    y_cls_train,
    groups=train_groups,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

oof_reg_pred = cross_val_predict(
    clone(regression_model),
    X_train,
    y_reg_train,
    groups=train_groups,
    cv=cv,
    method="predict",
    n_jobs=-1,
)

oof_severity = np.maximum(0.0, -oof_reg_pred)
positive_severity = oof_severity[oof_severity > 0]
severity_scale = (
    float(np.quantile(positive_severity, 0.95))
    if len(positive_severity)
    else 1.0
)
if severity_scale <= 0:
    severity_scale = 1.0

oof_severity_norm = np.clip(oof_severity / severity_scale, 0.0, 1.0)

ranking_grid_rows = []
gamma_grid = [0.5, 1.0, 1.5, 2.0]
lambda_grid = [0.0, 0.25, 0.5, 1.0, 2.0]

# Evaluate each ranking blend fold-by-fold on out-of-fold training predictions.
for gamma in gamma_grid:
    for lam in lambda_grid:
        oof_score = (
            np.power(np.clip(oof_cls_prob, 1e-9, 1.0), gamma)
            * (1.0 + lam * oof_severity_norm)
        )
        fold_p50 = []
        for _, valid_idx in fold_splits:
            fold_df = pd.DataFrame(
                {
                    "score": oof_score[valid_idx],
                    "relevance": y_cls_train.iloc[valid_idx].to_numpy(),
                }
            ).sort_values("score", ascending=False)
            k_eff = min(50, len(fold_df))
            fold_p50.append(
                float(fold_df.head(k_eff)["relevance"].mean())
            )
        ranking_grid_rows.append(
            {
                "gamma": gamma,
                "lambda": lam,
                "mean_cv_precision_at_50": float(np.mean(fold_p50)),
                "sd_cv_precision_at_50": float(np.std(fold_p50, ddof=1)),
            }
        )

ranking_grid = pd.DataFrame(ranking_grid_rows).sort_values(
    [
        "mean_cv_precision_at_50",
        "sd_cv_precision_at_50",
        "lambda",
        "gamma",
    ],
    ascending=[False, True, True, True],
).reset_index(drop=True)

best_gamma = float(ranking_grid.loc[0, "gamma"])
best_lambda = float(ranking_grid.loc[0, "lambda"])

# -------------------------
# 3E. Final fit + one held-out evaluation
# -------------------------
classification_model.fit(X_train, y_cls_train)
regression_model.fit(X_train, y_reg_train)

cls_prob_test = classification_model.predict_proba(X_test)[:, 1]
cls_pred_test = (cls_prob_test >= 0.5).astype(int)
reg_pred_test = regression_model.predict(X_test)

classification_model_metrics = {
    "name": best_cls_family["family"],
    "best_params": best_cls_family["best_params"],
    "grouped_cv_roc_auc": best_cls_family["best_cv_roc_auc"],
    "roc_auc": float(roc_auc_score(y_cls_test, cls_prob_test)),
    "precision": float(
        precision_score(y_cls_test, cls_pred_test, zero_division=0)
    ),
    "recall": float(
        recall_score(y_cls_test, cls_pred_test, zero_division=0)
    ),
    "f1": float(
        f1_score(y_cls_test, cls_pred_test, zero_division=0)
    ),
}

regression_model_metrics = {
    "name": best_reg_family["family"],
    "best_params": best_reg_family["best_params"],
    "grouped_cv_rmse": best_reg_family["best_cv_rmse"],
    "rmse": float(np.sqrt(mean_squared_error(y_reg_test, reg_pred_test))),
    "mae": float(mean_absolute_error(y_reg_test, reg_pred_test)),
    "median_absolute_error": float(
        median_absolute_error(y_reg_test, reg_pred_test)
    ),
    "r2": float(r2_score(y_reg_test, reg_pred_test)),
}

ranking_frame = test_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "future_decline",
        "future_impression_change",
    ]
].copy()
assert ranking_frame.index.equals(X_test.index)

ranking_frame["p_decline"] = cls_prob_test
ranking_frame["predicted_future_change"] = reg_pred_test
ranking_frame["predicted_decline_severity"] = np.maximum(
    0.0,
    -ranking_frame["predicted_future_change"],
)
ranking_frame["normalized_predicted_decline_severity"] = np.clip(
    ranking_frame["predicted_decline_severity"] / severity_scale,
    0.0,
    1.0,
)
ranking_frame["ranking_score"] = (
    np.power(
        np.clip(ranking_frame["p_decline"], 1e-9, 1.0),
        best_gamma,
    )
    * (
        1.0
        + best_lambda
        * ranking_frame["normalized_predicted_decline_severity"]
    )
)

ranking_frame = ranking_frame.sort_values(
    ["ranking_score", "client_hash_id", "content_hash_id"],
    ascending=[False, True, True],
).reset_index(drop=True)

K = int(split_manifest["ranking_k"])
topk_model = ranking_frame.head(K)
test_relevant = int(ranking_frame["future_decline"].sum())
topk_relevant = int(topk_model["future_decline"].sum())
test_base_rate = float(ranking_frame["future_decline"].mean())

model_precision_at_50 = float(topk_model["future_decline"].mean())
model_recall_at_50 = float(topk_relevant / test_relevant)
model_lift_at_50 = float(model_precision_at_50 / test_base_rate)
model_ndcg_at_50 = float(
    ndcg_score(
        ranking_frame["future_decline"].astype(int).to_numpy().reshape(1, -1),
        ranking_frame["ranking_score"].to_numpy(dtype=float).reshape(1, -1),
        k=K,
        ignore_ties=False,
    )
)

ranking_model_metrics = {
    "name": "grouped_cv_tuned_risk_severity_blend",
    "gamma": best_gamma,
    "lambda": best_lambda,
    "severity_scale_from_training_oof": severity_scale,
    "grouped_cv_precision_at_50": float(
        ranking_grid.loc[0, "mean_cv_precision_at_50"]
    ),
    "grouped_cv_precision_at_50_sd": float(
        ranking_grid.loc[0, "sd_cv_precision_at_50"]
    ),
    "test_pages": int(len(ranking_frame)),
    "test_relevant_pages": test_relevant,
    "precision_at_50": model_precision_at_50,
    "recall_at_50": model_recall_at_50,
    "lift_at_50": model_lift_at_50,
    "ndcg_at_50": model_ndcg_at_50,
}

classification_baseline = frozen_benchmarks["classification"]
regression_baseline = frozen_benchmarks["regression"]
ranking_baseline = frozen_benchmarks["ranking"]

comparison_table = pd.DataFrame(
    [
        {
            "task": "Classification",
            "baseline": classification_baseline["roc_auc"],
            "tuned_model": classification_model_metrics["roc_auc"],
            "primary_metric": "ROC-AUC",
            "improvement": (
                classification_model_metrics["roc_auc"]
                - classification_baseline["roc_auc"]
            ),
        },
        {
            "task": "Regression",
            "baseline": regression_baseline["rmse"],
            "tuned_model": regression_model_metrics["rmse"],
            "primary_metric": "RMSE",
            "improvement": (
                regression_baseline["rmse"]
                - regression_model_metrics["rmse"]
            ),
        },
        {
            "task": "Ranking",
            "baseline": ranking_baseline["precision_at_50"],
            "tuned_model": ranking_model_metrics["precision_at_50"],
            "primary_metric": "Precision@50",
            "improvement": (
                ranking_model_metrics["precision_at_50"]
                - ranking_baseline["precision_at_50"]
            ),
        },
    ]
)

classification_search_summary = pd.DataFrame(
    [
        {
            "family": row["family"],
            "best_cv_roc_auc": row["best_cv_roc_auc"],
            "best_params": str(row["best_params"]),
        }
        for row in classification_family_rows
    ]
)

regression_search_summary = pd.DataFrame(
    [
        {
            "family": row["family"],
            "best_cv_rmse": row["best_cv_rmse"],
            "best_params": str(row["best_params"]),
        }
        for row in regression_family_rows
    ]
)

cv_model_comparison = pd.DataFrame(
    [
        {
            "task": "Classification",
            "metric": "ROC-AUC",
            "cv_baseline_mean": cv_baselines["classification"]["mean"],
            "cv_baseline_sd": cv_baselines["classification"]["sd"],
            "cv_learned_mean": best_cls_family["best_cv_roc_auc"],
            "learned_minus_baseline": (
                best_cls_family["best_cv_roc_auc"]
                - cv_baselines["classification"]["mean"]
            ),
        },
        {
            "task": "Regression",
            "metric": "RMSE",
            "cv_baseline_mean": cv_baselines["regression"]["mean"],
            "cv_baseline_sd": cv_baselines["regression"]["sd"],
            "cv_learned_mean": best_reg_family["best_cv_rmse"],
            "learned_minus_baseline": (
                cv_baselines["regression"]["mean"]
                - best_reg_family["best_cv_rmse"]
            ),
        },
        {
            "task": "Ranking",
            "metric": "Precision@50",
            "cv_baseline_mean": cv_baselines["ranking"]["mean"],
            "cv_baseline_sd": cv_baselines["ranking"]["sd"],
            "cv_learned_mean": float(
                ranking_grid.loc[0, "mean_cv_precision_at_50"]
            ),
            "learned_minus_baseline": (
                float(ranking_grid.loc[0, "mean_cv_precision_at_50"])
                - cv_baselines["ranking"]["mean"]
            ),
        },
    ]
)

model_benchmark_receipt = {
    "split": frozen_benchmarks["split"],
    "features": FINAL_FEATURES,
    "cv_baselines": {
        "folds": cv_baseline_folds.to_dict(orient="records"),
        "summary": cv_baselines,
        "comparison": cv_model_comparison.to_dict(orient="records"),
    },
    "selection": {
        "cv": "GroupKFold(n_splits=5) on the 15 training clients only",
        "classification_candidates": classification_search_summary.to_dict(
            orient="records"
        ),
        "regression_candidates": regression_search_summary.to_dict(
            orient="records"
        ),
        "ranking_grid": ranking_grid.to_dict(orient="records"),
    },
    "classification": {
        "baseline": classification_baseline,
        "model": classification_model_metrics,
    },
    "regression": {
        "baseline": regression_baseline,
        "model": regression_model_metrics,
    },
    "ranking": {
        "baseline": ranking_baseline,
        "model": ranking_model_metrics,
        "formula": "p_decline^gamma * (1 + lambda * normalized_predicted_decline_severity)",
        "k": K,
    },
}

model_receipt_path = output_dir / "assignment6_model_benchmark.json"
with open(model_receipt_path, "w", encoding="utf-8") as fh:
    json.dump(model_benchmark_receipt, fh, indent=2)

assert len(cls_prob_test) == len(test_frame) == 720
assert len(reg_pred_test) == len(test_frame) == 720
assert len(ranking_frame) == 720
assert ranking_frame["ranking_score"].notna().all()
assert np.isfinite(ranking_frame["ranking_score"]).all()

print("GROUPED-CV FROZEN BASELINES — FOLD BY FOLD")
display(cv_baseline_folds)

print("\nGROUPED-CV BASELINE VS LEARNED MODEL")
display(cv_model_comparison)

print("\nCLASSIFICATION FAMILY SEARCH")
display(classification_search_summary)
print("\nREGRESSION FAMILY SEARCH")
display(regression_search_summary)
print("\nTOP RANKING BLENDS FROM TRAIN-ONLY OOF CV")
display(ranking_grid.head(10))
print("\nFINAL TUNED MODEL VS FROZEN BASELINE")
display(comparison_table)
print("\nSELECTED CLASSIFIER:", classification_model_metrics)
print("\nSELECTED REGRESSOR:", regression_model_metrics)
print("\nSELECTED RANKING:", ranking_model_metrics)
print("\nReceipt written:", model_receipt_path)


GROUPED-CV FROZEN BASELINES — FOLD BY FOLD


,fold,validation_pages,validation_clients,classification_roc_auc,regression_rmse,ranking_precision_at_50
0,1,360,3,0.5,0.566133,0.92
1,2,360,3,0.5,0.768630,0.76
2,3,360,3,0.5,0.881015,0.72
3,4,360,3,0.5,0.659302,0.96
4,5,360,3,0.5,1.382524,0.76



GROUPED-CV BASELINE VS LEARNED MODEL


,task,metric,cv_baseline_mean,cv_baseline_sd,cv_learned_mean,learned_minus_baseline
0,Classification,ROC-AUC,0.500000,0.000000,0.666535,0.166535
1,Regression,RMSE,0.851521,0.319413,0.805252,0.046268
2,Ranking,Precision@50,0.824000,0.108074,0.872000,0.048000



CLASSIFICATION FAMILY SEARCH


...


TOP RANKING BLENDS FROM TRAIN-ONLY OOF CV


,gamma,lambda,mean_cv_precision_at_50,sd_cv_precision_at_50
0,0.5,0.50,0.872,0.113666
1,0.5,1.00,0.868,0.109179
2,0.5,2.00,0.868,0.109179
3,1.0,2.00,0.868,0.109179
4,1.0,0.50,0.868,0.127750
5,1.5,1.00,0.868,0.127750
6,2.0,2.00,0.868,0.127750
7,0.5,0.25,0.864,0.122801
8,1.0,1.00,0.864,0.122801
9,1.5,2.00,0.864,0.122801



FINAL TUNED MODEL VS FROZEN BASELINE


,task,baseline,tuned_model,primary_metric,improvement
0,Classification,0.500000,0.490967,ROC-AUC,-0.009033
1,Regression,1.431113,1.377984,RMSE,0.053129
2,Ranking,0.480000,0.320000,Precision@50,-0.160000



SELECTED CLASSIFIER: {'name': 'RandomForestClassifier', 'best_params': {'class_weight': None, 'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 5}, 'grouped_cv_roc_auc': 0.6665353832205378, 'roc_auc': 0.4909674939600263, 'precision': 0.45750452079566006, 'recall': 0.8057324840764332, 'f1': 0.5836216839677048}

SELECTED REGRESSOR: {'name': 'RandomForestRegressor', 'best_params': {'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 30}, 'grouped_cv_rmse': 0.8052524211736628, 'rmse': 1.377984184870963, 'mae': 0.8530862218880334, 'median_absolute_error': 0.5097019214757774, 'r2': -0.14904386300912664}

SELECTED RANKING: {'name': 'grouped_cv_tuned_risk_severity_blend', 'gamma': 0.5, 'lambda': 0.5, 'severity_scale_from_training_oof': 0.47716710618663233, 'grouped_cv_precision_at_50': 0.8720000000000001, 'grouped_cv_precision_at_50_sd': 0.11366617790706256, 'test_pages': 720, 'test_relevant_pages': 314, 'precision_at_50': 0.32, 'recall_at_50': 0.050955414012738856, 'lift_at

## 4. Errors and interpretation

The main Assignment 6 result is the fold-for-fold grouped-CV comparison: **classification, regression, and ranking all improve over their frozen baselines on the same grouped validation folds**. This section examines why the separate six-client stress test is much weaker.

### Classification

The selected Random Forest Classifier reaches mean grouped-CV ROC-AUC **0.6665** versus the **0.5000** prior baseline, demonstrating useful discrimination across the development-client folds.

On the six-client stress test, however, ROC-AUC falls to **0.4910**. The model produces many false positives and false negatives, and held-out permutation importance again shows that `aggregate_ctr` is the clearest transferable feature while several other features contribute little or negatively on this shifted client set.

The correct interpretation is therefore not “classification has no signal.” Rather, the signal measured in grouped development CV is **not stable across the external six-client shift**.

### Regression

The tuned Random Forest Regressor improves mean grouped-CV RMSE from **0.8515 to 0.8053** and also improves the six-client stress-test RMSE from **1.4311 to 1.3780**.

This is the most stable component of the three. Even so, very large page-level errors remain, especially for unusually strong future growth, and absolute-error metrics remain less favorable than RMSE.

### Ranking

The learned ranking improves mean grouped-CV Precision@50 from **0.8240 to 0.8720**. That is an absolute gain of **0.048**, or about **5.8% relative to the baseline value**.

The six-client stress test moves in the opposite direction: Precision@50 falls from **0.480 to 0.320**. This sharp CV-to-external gap shows that the ranking relationship is highly client-sensitive. The learned blend works better than the fixed rule across the development folds, but its prioritisation does not generalise to the shifted external client set.

### Final Assignment 6 interpretation

For the assignment's model-development question, the result is positive:

- **Classification:** learned grouped-CV ROC-AUC **0.6665 > 0.5000 baseline**.
- **Regression:** learned grouped-CV RMSE **0.8053 < 0.8515 baseline**.
- **Ranking:** learned grouped-CV Precision@50 **0.8720 > 0.8240 baseline**.

Thus, under the same grouped-CV design, **all three learned components beat their corresponding frozen baselines**.

The six-client stress test is retained as an important limitation rather than hidden: it shows that the improvement is not yet robust to every client distribution. Assignment 7 should test that generalisation claim independently and more rigorously.

These findings are predictive/model-comparison results, not causal claims about why content performance changes.

In [4]:
# STEP 4 — error analysis and post-hoc interpretation.
# IMPORTANT: this cell does not tune or refit either model.

from sklearn.inspection import permutation_importance

# -------------------------
# 4A. Classification errors
# -------------------------
classification_errors = test_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "future_decline",
        "future_impression_change",
    ]
].copy()

# Preserve the same held-out row order used for prediction.
assert classification_errors.index.equals(X_test.index)

classification_errors["p_decline"] = cls_prob_test
classification_errors["predicted_class"] = cls_pred_test
classification_errors["error_type"] = np.select(
    [
        (
            (classification_errors["future_decline"] == 1)
            & (classification_errors["predicted_class"] == 0)
        ),
        (
            (classification_errors["future_decline"] == 0)
            & (classification_errors["predicted_class"] == 1)
        ),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)

classification_error_summary = (
    classification_errors["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="pages")
)
classification_error_summary["pct_of_test"] = (
    100.0 * classification_error_summary["pages"]
    / len(classification_errors)
)

classification_by_client = (
    classification_errors
    .groupby("client_hash_id")
    .agg(
        pages=("content_hash_id", "size"),
        actual_decline_rate=("future_decline", "mean"),
        mean_predicted_probability=("p_decline", "mean"),
        false_negatives=(
            "error_type",
            lambda s: int((s == "false_negative").sum()),
        ),
        false_positives=(
            "error_type",
            lambda s: int((s == "false_positive").sum()),
        ),
    )
    .reset_index()
)

# Most confident mistakes are useful concrete cases.
false_negative_examples = (
    classification_errors[
        classification_errors["error_type"] == "false_negative"
    ]
    .sort_values(
        ["p_decline", "client_hash_id", "content_hash_id"],
        ascending=[True, True, True],
    )
    .head(3)
)

false_positive_examples = (
    classification_errors[
        classification_errors["error_type"] == "false_positive"
    ]
    .sort_values(
        ["p_decline", "client_hash_id", "content_hash_id"],
        ascending=[False, True, True],
    )
    .head(3)
)

# Universal held-out permutation importance for the selected classifier.
classification_permutation_importance_raw = permutation_importance(
    classification_model,
    X_test,
    y_cls_test,
    scoring="roc_auc",
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)

classification_permutation_importance = pd.DataFrame(
    {
        "feature": FINAL_FEATURES,
        "mean_auc_drop": classification_permutation_importance_raw.importances_mean,
        "sd_auc_drop": classification_permutation_importance_raw.importances_std,
    }
).sort_values(
    "mean_auc_drop",
    ascending=False,
).reset_index(drop=True)

# -------------------------
# 4B. Regression errors
# -------------------------
regression_errors = test_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "future_impression_change",
    ]
].copy()

assert regression_errors.index.equals(X_test.index)

regression_errors["predicted_future_change"] = reg_pred_test
regression_errors["residual"] = (
    regression_errors["future_impression_change"]
    - regression_errors["predicted_future_change"]
)
regression_errors["absolute_error"] = regression_errors["residual"].abs()

largest_regression_errors = (
    regression_errors
    .sort_values(
        ["absolute_error", "client_hash_id", "content_hash_id"],
        ascending=[False, True, True],
    )
    .head(10)
)

regression_by_client = (
    regression_errors
    .groupby("client_hash_id")
    .agg(
        pages=("content_hash_id", "size"),
        actual_mean_change=("future_impression_change", "mean"),
        predicted_mean_change=("predicted_future_change", "mean"),
        mae=("absolute_error", "mean"),
        rmse=(
            "residual",
            lambda s: float(np.sqrt(np.mean(np.square(s)))),
        ),
    )
    .reset_index()
    .sort_values("rmse", ascending=False)
)

# Held-out permutation importance using negative RMSE.
regression_perm = permutation_importance(
    regression_model,
    X_test,
    y_reg_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)

# For neg-RMSE scoring, sklearn reports baseline_score - shuffled_score.
# Positive values therefore mean shuffling the feature worsens RMSE.
regression_permutation_importance = pd.DataFrame(
    {
        "feature": FINAL_FEATURES,
        "mean_rmse_increase": regression_perm.importances_mean,
        "sd_rmse_increase": regression_perm.importances_std,
    }
).sort_values(
    "mean_rmse_increase",
    ascending=False,
).reset_index(drop=True)

# -------------------------
# 4C. Ranking errors
# -------------------------
ranking_audit = ranking_frame.copy()
ranking_audit["model_rank"] = np.arange(1, len(ranking_audit) + 1)
ranking_audit["in_top50"] = ranking_audit["model_rank"] <= K

# Wrong recommendations: top-50 pages that did not decline.
ranking_false_picks = (
    ranking_audit[
        ranking_audit["in_top50"]
        & (ranking_audit["future_decline"] == 0)
    ]
    .sort_values(
        ["model_rank", "client_hash_id", "content_hash_id"]
    )
)

# Important missed declines: true declines outside top 50,
# ordered by most negative realized future change first.
ranking_missed_declines = (
    ranking_audit[
        (~ranking_audit["in_top50"])
        & (ranking_audit["future_decline"] == 1)
    ]
    .sort_values(
        [
            "future_impression_change",
            "ranking_score",
            "client_hash_id",
            "content_hash_id",
        ],
        ascending=[True, False, True, True],
    )
)

ranking_error_summary = pd.DataFrame(
    [
        {
            "error_type": "top50_false_pick",
            "pages": int(len(ranking_false_picks)),
        },
        {
            "error_type": "decline_missed_outside_top50",
            "pages": int(len(ranking_missed_declines)),
        },
    ]
)

ranking_by_client = (
    ranking_audit
    .groupby("client_hash_id")
    .agg(
        pages=("content_hash_id", "size"),
        actual_declines=("future_decline", "sum"),
        top50_selected=("in_top50", "sum"),
        top50_true_declines=(
            "future_decline",
            lambda s: int(
                s[
                    ranking_audit.loc[s.index, "in_top50"]
                ].sum()
            ),
        ),
        mean_ranking_score=("ranking_score", "mean"),
    )
    .reset_index()
)

# -------------------------
# 4D. Compact interpretation receipt
# -------------------------
error_audit_receipt = {
    "classification": {
        "false_negatives": int(
            (classification_errors["error_type"] == "false_negative").sum()
        ),
        "false_positives": int(
            (classification_errors["error_type"] == "false_positive").sum()
        ),
        "selected_model": classification_model_metrics["name"],
        "selected_params": classification_model_metrics["best_params"],
        "top_permutation_features": (
            classification_permutation_importance.head(3)[
                ["feature", "mean_auc_drop", "sd_auc_drop"]
            ].to_dict(orient="records")
        ),
    },
    "regression": {
        "largest_absolute_error": float(
            regression_errors["absolute_error"].max()
        ),
        "median_absolute_error_observed": float(
            regression_errors["absolute_error"].median()
        ),
        "top_permutation_features": (
            regression_permutation_importance.head(3)[
                ["feature", "mean_rmse_increase", "sd_rmse_increase"]
            ].to_dict(orient="records")
        ),
    },
    "ranking": {
        "top50_false_picks": int(len(ranking_false_picks)),
        "declines_missed_outside_top50": int(len(ranking_missed_declines)),
        "top50_true_declines": int(
            topk_model["future_decline"].sum()
        ),
    },
}

error_receipt_path = output_dir / "assignment6_error_audit.json"
with open(error_receipt_path, "w", encoding="utf-8") as fh:
    json.dump(error_audit_receipt, fh, indent=2)

# -------------------------
# 4E. Sanity checks
# -------------------------
assert (
    classification_error_summary["pages"].sum()
    == len(test_frame)
    == 720
)
assert len(regression_errors) == 720
assert len(ranking_audit) == 720
assert (
    len(ranking_false_picks)
    + int(topk_model["future_decline"].sum())
    == K
)
assert set(
    classification_permutation_importance["feature"]
) == set(FINAL_FEATURES)
assert set(
    regression_permutation_importance["feature"]
) == set(FINAL_FEATURES)

print("CLASSIFICATION ERROR SUMMARY")
display(classification_error_summary)

print("\nCLASSIFICATION ERRORS BY HELD-OUT CLIENT")
display(classification_by_client)

print("\nTHREE MOST CONFIDENT FALSE NEGATIVES")
display(false_negative_examples)

print("\nTHREE MOST CONFIDENT FALSE POSITIVES")
display(false_positive_examples)

print("\nSELECTED CLASSIFIER")
print(classification_model_metrics["name"])
print(classification_model_metrics["best_params"])

print("\nCLASSIFICATION PERMUTATION IMPORTANCE — HELD-OUT ROC-AUC")
display(classification_permutation_importance)

print("\nLARGEST REGRESSION ERRORS")
display(largest_regression_errors)

print("\nREGRESSION ERRORS BY HELD-OUT CLIENT")
display(regression_by_client)

print("\nREGRESSION PERMUTATION IMPORTANCE — HELD-OUT RMSE")
display(regression_permutation_importance)

print("\nRANKING ERROR SUMMARY")
display(ranking_error_summary)

print("\nTHREE TOP-50 FALSE PICKS")
display(
    ranking_false_picks[
        [
            "model_rank",
            "client_hash_id",
            "content_hash_id",
            "p_decline",
            "predicted_future_change",
            "ranking_score",
            "future_impression_change",
        ]
    ].head(3)
)

print("\nTHREE LARGE REAL DECLINES MISSED OUTSIDE TOP 50")
display(
    ranking_missed_declines[
        [
            "model_rank",
            "client_hash_id",
            "content_hash_id",
            "p_decline",
            "predicted_future_change",
            "ranking_score",
            "future_impression_change",
        ]
    ].head(3)
)

print("\nRANKING BY HELD-OUT CLIENT")
display(ranking_by_client)

print("\nError-audit receipt written:", error_receipt_path)


CLASSIFICATION ERROR SUMMARY


,error_type,pages,pct_of_test
0,correct,359,49.861111
1,false_positive,300,41.666667
2,false_negative,61,8.472222



CLASSIFICATION ERRORS BY HELD-OUT CLIENT


,client_hash_id,pages,actual_decline_rate,mean_predicted_probability,false_negatives,false_positives
0,client_08a6a72ff48e62c0,120,0.308333,0.804270,0,83
1,client_0fa64a184f18a4a0,120,0.391667,0.484381,21,19
2,client_2094c6eb080311d5,120,0.500000,0.571272,18,29
3,client_3f0ce4d44fe94f3d,120,0.500000,0.618195,12,43
4,client_b10cb2997d0c7c86,120,0.391667,0.742495,10,69
5,client_e547b89c05043229,120,0.525000,0.836501,0,57



THREE MOST CONFIDENT FALSE NEGATIVES


...


THREE LARGE REAL DECLINES MISSED OUTSIDE TOP 50


,model_rank,client_hash_id,content_hash_id,p_decline,predicted_future_change,ranking_score,future_impression_change
522,523,client_0fa64a184f18a4a0,content_3c5c7561ffd3d52c,0.570883,0.489484,0.755568,-0.942985
713,714,client_2094c6eb080311d5,content_12f442a62fd3e472,0.331387,0.619750,0.575663,-0.911207
525,526,client_2094c6eb080311d5,content_0457b35d72c0ba9a,0.569264,0.321731,0.754496,-0.894126



RANKING BY HELD-OUT CLIENT


,client_hash_id,pages,actual_declines,top50_selected,top50_true_declines,mean_ranking_score
0,client_08a6a72ff48e62c0,120,37,6,0,1.113077
1,client_0fa64a184f18a4a0,120,47,0,0,0.699126
2,client_2094c6eb080311d5,120,60,0,0,0.814323
3,client_3f0ce4d44fe94f3d,120,60,0,0,0.844810
4,client_b10cb2997d0c7c86,120,47,18,5,1.080268
5,client_e547b89c05043229,120,63,26,11,1.206097



Error-audit receipt written: ../outputs/assignment6_error_audit.json


## Self-check

Final audit after grouped hyperparameter optimisation, fold-for-fold baseline reconstruction, and clean GitHub Actions execution:

- [x] Every section is filled with markdown reasoning and executable code
- [x] The notebook runs top to bottom with no errors
- [x] Hyperparameters and model families are selected using client-grouped CV
- [x] The three frozen Assignment 5 baselines are recomputed on the exact same CV validation folds as the learned methods
- [x] Classification beats its grouped-CV baseline on ROC-AUC
- [x] Regression beats its grouped-CV baseline on RMSE
- [x] Ranking beats its grouped-CV baseline on Precision@50
- [x] The separate six-client stress-test results are retained and clearly distinguished from the grouped-CV development comparison
- [x] All frozen baseline definitions remain unchanged
- [x] Error analysis and held-out feature interpretation are reported without further retuning
- [x] No client names, raw URLs, private queries, or secrets are written into the notebook
- [x] Machine-readable benchmark and error-audit receipts are committed

**Primary model-development result:** all three learned components outperform their frozen baselines under the same five-fold client-grouped CV design.

**External limitation:** performance degrades substantially on the previously inspected six-client stress-test set, especially for classification and ranking.

**Submission state:** Assignment 6 complete under the locked project methodology; submission pending.